# SuperSimpleNet — Augmentation ablation runner

Runs **every** augmentation config in `configs/*.json` (or a chosen subset), one
Drive folder per config. **Stop/resume-safe**: on restart it skips any config
that already finished (a `DONE.txt` marker), so you never lose completed work.

Workflow:
1. Run **Setup** once (mount Drive, clone your branch, install deps).
2. (SUP mode only) Run **Dataset prep** once to seed anomalies into the train set.
3. Run the **Grid runner** — re-run it any time; it resumes where it left off.
4. Run **Summary** to collect all metrics into one CSV on Drive.


## 1) Setup — mount Drive, clone branch, install dependencies

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# ========================= CONFIGURE ME =========================
GIT_REPO_URL = 'https://github.com/EmanuelePietroCometti/SuperSimpleNet.git'
# IMPORTANT: your augmentation branch (must carry aug_config.py / augment_ssn.py / configs/).
BRANCH_NAME  = 'baseline_prove'
REPO_PATH    = '/content/SuperSimpleNet'

DATA_PATH    = '/content/drive/MyDrive/Tesi/MVTec'
# Base Drive folder for the WHOLE ablation grid (one sub-folder per config is created here).
ABLATION_BASE = '/content/drive/MyDrive/Tesi/Risultati_SSN/ablation_carpet'

CATEGORY = 'carpet'
MODE     = 'sup'          # 'sup' (supervised/mixed) or 'unsup'

# Shared training hyperparameters (identical across configs so only augmentation varies)
EPOCHS = 100
BATCH  = 4
SEED   = 42
# ===============================================================

if not os.path.exists(REPO_PATH):
    print(f">>> Cloning branch '{BRANCH_NAME}'...")
    !git clone -b {BRANCH_NAME} {GIT_REPO_URL} {REPO_PATH}
else:
    print(f">>> Repo present. Checking out '{BRANCH_NAME}' and pulling...")
    os.chdir(REPO_PATH)
    !git checkout {BRANCH_NAME}
    !git pull origin {BRANCH_NAME}

os.chdir(REPO_PATH)
os.makedirs(ABLATION_BASE, exist_ok=True)
print("Working dir:", os.getcwd())
print("Ablation base:", ABLATION_BASE)

!pip install tqdm numpy==1.26.0 anomalib==0.7
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 --extra-index-url https://download.pytorch.org/whl/cu118
!pip install wandb optuna


## 2) Dataset prep (SUP mode only — run once)

Seeds a few anomalous samples (with masks) into the train set, exactly like the
main notebook. Idempotent: safe to re-run. Skip entirely for `unsup` mode.

In [ ]:
import os, glob, shutil

NUM_ANOMALIES_PER_DEFECT = 2

if MODE == 'sup':
    dataset_root = os.path.join(DATA_PATH, CATEGORY)
    test_dir  = os.path.join(dataset_root, 'test')
    gt_root   = os.path.join(dataset_root, 'ground_truth')
    train_dir = os.path.join(dataset_root, 'train')

    if not os.path.exists(test_dir):
        raise FileNotFoundError(f"Cannot find test folder: {test_dir}")

    defect_types = [d for d in os.listdir(test_dir)
                    if os.path.isdir(os.path.join(test_dir, d)) and d != 'good']
    print(f">>> SUP prep for '{CATEGORY}', defects: {defect_types}")

    for defect in defect_types:
        defect_test_dir = os.path.join(test_dir, defect)
        defect_gt_dir   = os.path.join(gt_root, defect)
        target_train_defect_dir = os.path.join(train_dir, defect)
        target_train_gt_dir     = os.path.join(train_dir, 'ground_truth', defect)
        os.makedirs(target_train_defect_dir, exist_ok=True)
        os.makedirs(target_train_gt_dir, exist_ok=True)

        images = sorted(glob.glob(os.path.join(defect_test_dir, "*.png")))
        for img_path in images[:NUM_ANOMALIES_PER_DEFECT]:
            stem = os.path.splitext(os.path.basename(img_path))[0]
            dest_img = os.path.join(target_train_defect_dir, os.path.basename(img_path))
            if not os.path.exists(dest_img):
                shutil.copy(img_path, dest_img)
            for mask_path in (os.path.join(defect_gt_dir, f"{stem}_mask.png"),
                              os.path.join(defect_gt_dir, f"{stem}.png")):
                if os.path.exists(mask_path):
                    dest_mask = os.path.join(target_train_gt_dir, os.path.basename(mask_path))
                    if not os.path.exists(dest_mask):
                        shutil.copy(mask_path, dest_mask)
                    break
    print(">>> SUP dataset prep done.")
else:
    print(">>> MODE is not 'sup' -> skipping dataset prep.")


## 3) Grid runner (resume-safe)

Runs each config with identical hyperparameters, saving to its own Drive folder
`ABLATION_BASE/<CATEGORY>__<config>/`. A config that already has a `DONE.txt` is
**skipped**, so you can stop and re-run this cell freely.

To run a subset, set `CONFIG_FILES` manually (e.g. only the single-family ones).

In [ ]:
import os, glob, json, shutil, subprocess

os.chdir(REPO_PATH)

# ===================== CONFIGS TO RUN (edit freely) =====================
# Explicit list, grouped as in configs/EXPERIMENTS.md. Comment out any row you
# don't want, or reorder. To instead run EVERY json present automatically, use
# the glob line at the bottom.
CONFIG_FILES = [
    # -- Block A: reference --
    'configs/aug_off.json',            # baseline, no augmentation
    # -- Block B: single-family --
    'configs/aug_hflip.json',
    'configs/aug_vflip.json',
    'configs/aug_affine.json',
    'configs/aug_colorjitter.json',
    'configs/aug_grayscale.json',
    'configs/aug_blur.json',
    'configs/aug_equalize.json',
    'configs/aug_speckle.json',
    'configs/aug_dynamic_crop.json',
    # -- Block C: family groups --
    'configs/aug_flips.json',
    'configs/aug_geometric.json',
    'configs/aug_photometric.json',
    'configs/aug_full.json',
    # -- Block D: intensity sweep (strong) --
    'configs/aug_colorjitter_strong.json',
    'configs/aug_affine_strong.json',
    'configs/aug_blur_strong.json',
    'configs/aug_speckle_strong.json',
    # -- Block E: combined presets --
    'configs/aug_light.json',
    'configs/aug_medium.json',
]
# Alternative: run every json present, no manual list:
# CONFIG_FILES = sorted(glob.glob('configs/*.json'))

# Keep only the ones that actually exist on disk (guards against typos/renames).
CONFIG_FILES = [c for c in CONFIG_FILES if os.path.exists(c)]
# =======================================================================

print(f"{len(CONFIG_FILES)} configs queued:")
for c in CONFIG_FILES:
    print("   -", os.path.basename(c))

for cfg_path in CONFIG_FILES:
    cfg_name = os.path.splitext(os.path.basename(cfg_path))[0]
    run_dir  = os.path.join(ABLATION_BASE, f"{CATEGORY}__{cfg_name}")
    done_marker = os.path.join(run_dir, "DONE.txt")

    if os.path.exists(done_marker):
        print(f"[SKIP] {cfg_name} already completed")
        continue

    os.makedirs(run_dir, exist_ok=True)
    print("\n" + "=" * 64)
    print(f"[RUN ] {cfg_name}  ->  {run_dir}")
    print("=" * 64)

    cmd = [
        "python", "train.py",
        "--dataset", "mvtec",
        "--category", CATEGORY,
        "--mode", MODE,
        "--data_path", DATA_PATH,
        "--datasets_folder", DATA_PATH,
        "--results_save_path", run_dir,
        "--setup_name", f"ssn_{cfg_name}",
        "--num_workers", "1",
        "--backbone", "wide_resnet50_2",
        "--layers", "layer2", "layer3",
        "--image_size", "512", "512",
        "--epochs", str(EPOCHS),
        "--batch", str(BATCH),
        "--perlin_thr", "0.2",
        "--noise_std", "0.015",
        "--seg_lr", "0.0002",
        "--dec_lr", "0.0002",
        "--adapt_lr", "0.0001",
        "--patch_size", "3",
        "--gamma", "0.4",
        "--eval_step_size", "5",
        "--seed", str(SEED),
        "--aug_config", cfg_path,
    ]

    # Stream output live; inherit stdout/stderr so tqdm shows in the cell.
    ret = subprocess.run(cmd)

    if ret.returncode == 0:
        shutil.copy(cfg_path, os.path.join(run_dir, "aug_config_used.json"))
        with open(done_marker, "w") as f:
            f.write("completed\n")
        print(f"[DONE] {cfg_name}")
    else:
        print(f"[FAIL] {cfg_name} (exit {ret.returncode}) -- no DONE marker, will retry next run")

print("\n>>> Grid runner finished this pass.")


## 4) Summary — collect every config's metrics into one CSV

In [ ]:
import os, glob, json
import pandas as pd

rows = []
for cfg_path in sorted(glob.glob('configs/*.json')):
    cfg_name = os.path.splitext(os.path.basename(cfg_path))[0]
    run_dir  = os.path.join(ABLATION_BASE, f"{CATEGORY}__{cfg_name}")
    metric_files = glob.glob(os.path.join(run_dir, "**", "metrics.json"), recursive=True)
    if not metric_files:
        rows.append({"config": cfg_name, "status": "not_run"})
        continue
    with open(sorted(metric_files)[0]) as f:
        m = json.load(f)
    row = {"config": cfg_name, "status": "done"}
    row.update(m)
    rows.append(row)

df = pd.DataFrame(rows)
# Put the most relevant metrics first when present
front = [c for c in ["config", "status", "I-AUROC", "P-AUROC", "AUPRO", "AP-loc",
                     "AP-det", "F1-score"] if c in df.columns]
df = df[front + [c for c in df.columns if c not in front]]

out_csv = os.path.join(ABLATION_BASE, f"ablation_summary_{CATEGORY}.csv")
df.to_csv(out_csv, index=False)
print(df.to_string(index=False))
print("\nSaved summary ->", out_csv)
